# Level 1 — Score a real GCS Zarr locally (no Coiled)

Reads from GCS, applies scoring, writes scored zarr back to GCS.
Uses local Dask threads — no Coiled cluster needed.

In [ ]:
import sys
sys.path.insert(0, "/home/stefan/CRS/CRS.ZarrPipelines/")
import xarray as xr
import matplotlib.pyplot as plt
from app.domain.pipeline import score_hazard

## Score HS (Heat Stress) — all three scales

Writes to `gs://crs_climate_data_public/production_test/hazard_scores/HS.zarr`.
Smart merge: existing scales are kept; only requested scales are replaced/appended.

In [ ]:
score_hazard("HS", scales=["5", "10", "100"])
print("Scoring complete")

## Inspect the scored zarr

In [ ]:
ds = xr.open_zarr("gs://crs_climate_data_public/production_test/hazard_scores/HS.zarr")
print(ds)
print("scoring dim:", ds.scoring.values)  # should include 5, 10, 100

In [ ]:
# Show available non-spatial coordinate values
for coord in ("scenario", "time", "statistic", "model"):
    if coord in ds.coords:
        print(f"{coord}: {ds[coord].values}")

## Visualise — East Africa slice, all three scales

In [ ]:
# Adjust sel values to match your zarr coordinate labels
sel = dict(scenario="RCP45", statistic="mean", time="Cc",
           lat=slice(-5, 15), lon=slice(30, 50))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (scale, cmap, vmax) in zip(axes, [("5", "YlOrRd", 5), ("10", "YlOrRd", 10), ("100", "plasma", 100)]):
    ds.sel(scoring=scale, **sel)["score"].compute().plot(
        ax=ax, cmap=cmap, vmin=0, vmax=vmax
    )
    ax.set_title(f"HS ({scale}-pt)")
plt.tight_layout()
plt.show()